# Spark DataFrames and Transformations


## Why DataFrames Matter

In real projects:
- You don’t create tiny lists by hand
- You **load** CSV / JSON / Parquet / tables at scale

👉 **DataFrames** are Spark’s main abstraction


## Prerequisites

- PySpark + JDK set up ([`README.md`](README.md)).
- Run this notebook from `notebooks/04_spark/` so outputs land next to the lesson.
- Direct `spark.read.json("https://...")` **often fails** (no Hadoop filesystem for HTTP). The load cell below downloads JSON locally first.


## Spark session


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("DataFrameTransforms")
    .master("local[*]")
    .getOrCreate()
)


## Load data (JSON)

JSONPlaceholder returns a **JSON array**. We save it to a temp file and read with `multiLine` — reliable across environments.

_If you already have a local file, point `spark.read.json(...)` at that path instead._


In [ ]:
import tempfile
import urllib.request
import os

url = "https://jsonplaceholder.typicode.com/users"
fd, path = tempfile.mkstemp(suffix=".json")
os.close(fd)

urllib.request.urlretrieve(url, path)

df = spark.read.option("multiLine", "true").json(path)

df.show(5, truncate=False)


## Schema understanding


In [ ]:
df.printSchema()


## Select columns


In [ ]:
df.select("name", "email").show()


## Filter data


In [ ]:
df.filter(df["id"] > 5).show()


## Add new column


In [ ]:
from pyspark.sql.functions import length

df2 = df.withColumn("name_length", length(df["name"]))
df2.show(5, truncate=False)


## Rename columns


In [ ]:
df3 = df.withColumnRenamed("name", "user_name")
df3.select("id", "user_name", "email").show(5, truncate=False)


## Drop columns


In [ ]:
df.drop("phone").select("id", "name", "email").show(5, truncate=False)


## Group by (aggregation)


In [ ]:
df.groupBy("company.name").count().orderBy("count", ascending=False).show(truncate=False)


## Sort data


In [ ]:
df.orderBy(df["id"].desc()).show(5, truncate=False)


## Real transformation pipeline


In [ ]:
from pyspark.sql.functions import col, upper

transformed = df.select(
    col("id"),
    upper(col("name")).alias("name"),
    col("email"),
)

transformed.show(5, truncate=False)


## Write data

Writes a **directory** of JSON part-files (`output_users/`) — normal for Spark.


In [ ]:
import shutil

out_dir = "output_users"
shutil.rmtree(out_dir, ignore_errors=True)

transformed.write.mode("overwrite").json(out_dir)

print(f"Wrote JSON dataset under: {out_dir}/")


## Practice

1. Filter users where `address.city` equals a city you pick (use `col("address.city")` or `df.filter(...)`).
2. Add column **`email_domain`** from `email` (split on `@` — `split` + `element_at`, or string functions).
3. **Count** users per `company.name`.


## Assignment

Build a small pipeline:

- Load JSON (same source or local file)
- **Clean:** uppercase names; drop rows with invalid email (no `@`)
- **Write** cleaned data to an output path

**Bonus:** write **Parquet** (`write.mode(...).parquet(...)`).


## Interview Questions

1. What is a Spark DataFrame?
2. Difference between `select` and `filter`?
3. What is schema?
4. How do you transform data in Spark?


## What you just learned

- Load **semi-structured** JSON at scale
- **Transform** with typed columns and nested fields
- **Write** partitioned outputs for downstream jobs

👉 Spark **ETL** in miniature.


---

**Next:** **Spark SQL** — query DataFrames with SQL and mix SQL + DataFrames.

**Reality check:** you can wire Python ETL + Spark for volume — now you’re in serious analytics-engineering territory.


## Optional: stop Spark


In [ ]:
spark.stop()
print("Spark stopped.")


## Your tasks

- [ ] Run **Spark session** → **Write data**; inspect `output_users/` (part-*.json).
- [ ] **Practice:** city filter, email domain column, counts by company.
- [ ] **Assignment:** clean + filter emails + write JSON; **bonus:** Parquet.
- [ ] Answer interview questions; explain **lazy** transforms vs **write** action.


## Shared dataset usage (`resources/datasets/`)

```python
users_df = spark.read.json("resources/datasets/users.json")
orders_df = spark.read.json("resources/datasets/orders.json")

tx_df = spark.read.csv("resources/datasets/transactions.csv", header=True, inferSchema=True)
large_orders = spark.read.csv("resources/datasets/large_orders.csv", header=True, inferSchema=True)

# Simulate scale quickly for local experiments
large_orders_scaled = large_orders.union(large_orders).union(large_orders).union(large_orders)
```

These files are reusable across SQL, ETL, Spark, and Kafka notebooks.